# IBM Granite Docling 258M Model Testing

This notebook tests the [IBM Granite Docling 258M](https://huggingface.co/ibm-granite/granite-docling-258M) model - a multimodal Image-Text-to-Text model for efficient document conversion.

## Features
- Enhanced equation and code recognition
- Flexible inference modes (full-page or region-based)
- Document element question-answering
- Experimental multilingual support (Japanese, Arabic, Chinese)

## Architecture
- Vision Encoder: SigLIP2
- Language Model: Granite 165M
- Base Architecture: Idefics3

## 1. Installation and Setup

In [ ]:
# Install required dependencies
!uv add -q torch transformers pillow accelerate

In [ ]:
# Import required libraries
import torch
from transformers import AutoProcessor, AutoModelForVision2Seq, AutoModelForImageTextToText
from PIL import Image
import requests
from pathlib import Path
import json

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## 2. Load Model and Processor

In [ ]:
# Model configuration
MODEL_ID = "ibm-granite/granite-docling-258M"

# Determine device and dtype
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if torch.cuda.is_available() else torch.float32

print(f"Loading model on {device} with {dtype}...")

In [ ]:
# Load processor
processor = AutoProcessor.from_pretrained(MODEL_ID)
print("✓ Processor loaded successfully")

In [ ]:
# Load model
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    device_map="auto",
    dtype=dtype
)
print("✓ Model loaded successfully")
print(f"Model device: {next(model.parameters()).device}")
print(f"Model dtype: {next(model.parameters()).dtype}")

## 3. Helper Functions

In [ ]:
def load_image(image_path_or_url: str) -> Image.Image:
    """
    Load image from local path or URL.
    
    Args:
        image_path_or_url: Local file path or HTTP(S) URL
    
    Returns:
        PIL Image object in RGB format
    """
    if image_path_or_url.startswith(('http://', 'https://')):
        response = requests.get(image_path_or_url, stream=True)
        response.raise_for_status()
        image = Image.open(response.raw)
    else:
        image = Image.open(image_path_or_url)
    
    return image.convert("RGB")


def extract_text_from_document(
    image: Image.Image,
    prompt: str = "Convert this document to Markdown format.",
    max_new_tokens: int = 2048,
    temperature: float = 0.3,
    do_sample: bool = True
) -> str:
    """
    Extract text from document image using Granite Docling model.
    
    Args:
        image: PIL Image object
        prompt: Instruction prompt for the model
        max_new_tokens: Maximum tokens to generate
        temperature: Sampling temperature (lower = more deterministic)
        do_sample: Whether to use sampling or greedy decoding
    
    Returns:
        Extracted text as string
    """
    # Prepare inputs
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": prompt}
            ]
        }
    ]
    
    # Apply chat template and tokenize
    inputs = processor(
        images=image,
        text=processor.apply_chat_template(
            messages,
            add_generation_prompt=True
        ),
        return_tensors="pt"
    ).to(model.device)
    
    # Generate
    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=do_sample
        )
    
    # Decode output, removing input tokens
    generated_text = processor.batch_decode(
        generated_ids[:, inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    )[0]
    
    return generated_text.strip()


def answer_document_question(
    image: Image.Image,
    question: str,
    max_new_tokens: int = 512,
    temperature: float = 0.3
) -> str:
    """
    Answer a question about the document content.
    
    Args:
        image: PIL Image object
        question: Question to answer
        max_new_tokens: Maximum tokens to generate
        temperature: Sampling temperature
    
    Returns:
        Answer as string
    """
    return extract_text_from_document(
        image=image,
        prompt=question,
        max_new_tokens=max_new_tokens,
        temperature=temperature
    )


print("✓ Helper functions defined")

## 4. Test with Local Images

Test the model with document images from the project directory.

In [ ]:
# List available test images in project directory
project_root = Path.cwd()
image_extensions = ('.png', '.jpg', '.jpeg')
test_images = [
    f for f in project_root.glob('*.{png,jpg,jpeg}')
    if f.suffix.lower() in image_extensions
]

print(f"Found {len(test_images)} test images:")
for img in sorted(test_images)[:10]:  # Show first 10
    print(f"  - {img.name}")

### 4.1 Test Invoice Extraction

In [ ]:
# Test with invoice image
invoice_path = "invoice_1.png"

if Path(invoice_path).exists():
    print(f"Loading {invoice_path}...")
    invoice_image = load_image(invoice_path)
    display(invoice_image.resize((400, int(400 * invoice_image.height / invoice_image.width))))  # Show resized preview
    
    print("\nExtracting invoice data to Markdown...")
    invoice_markdown = extract_text_from_document(
        image=invoice_image,
        prompt="Convert this invoice to Markdown format, preserving all details including line items, amounts, and totals.",
        max_new_tokens=2048
    )
    
    print("\n" + "="*80)
    print("EXTRACTED INVOICE (Markdown):")
    print("="*80)
    print(invoice_markdown)
else:
    print(f"Image not found: {invoice_path}")

### 4.2 Test Bank Statement Extraction

In [ ]:
# Test with bank statement image
statement_path = "bank_statement_1.png"

if Path(statement_path).exists():
    print(f"Loading {statement_path}...")
    statement_image = load_image(statement_path)
    display(statement_image.resize((400, int(400 * statement_image.height / statement_image.width))))
    
    print("\nExtracting bank statement data...")
    statement_markdown = extract_text_from_document(
        image=statement_image,
        prompt="Convert this bank statement to Markdown format, including all transactions, dates, and amounts.",
        max_new_tokens=2048
    )
    
    print("\n" + "="*80)
    print("EXTRACTED BANK STATEMENT (Markdown):")
    print("="*80)
    print(statement_markdown)
else:
    print(f"Image not found: {statement_path}")

## 5. Test Question Answering

Test the model's ability to answer specific questions about document content.

In [ ]:
# Test Q&A with invoice
if Path(invoice_path).exists():
    invoice_image = load_image(invoice_path)
    
    questions = [
        "What is the total amount on this invoice?",
        "What is the invoice number?",
        "What is the invoice date?",
        "Who is the vendor/seller?",
        "List all line items with their quantities and prices."
    ]
    
    print("Testing Question Answering on Invoice:\n")
    for i, question in enumerate(questions, 1):
        print(f"Q{i}: {question}")
        answer = answer_document_question(invoice_image, question, max_new_tokens=512)
        print(f"A{i}: {answer}")
        print()
else:
    print(f"Invoice image not found: {invoice_path}")

## 6. Test with Sample Documents from URL

Test with publicly available document images.

In [ ]:
# Test with a sample receipt image from web
sample_url = "https://raw.githubusercontent.com/tesseract-ocr/docs/master/img/receipts/example.png"

try:
    print(f"Loading sample image from URL...")
    sample_image = load_image(sample_url)
    display(sample_image.resize((400, int(400 * sample_image.height / sample_image.width))))
    
    print("\nExtracting document text...")
    extracted_text = extract_text_from_document(
        image=sample_image,
        prompt="Convert this document to clean, readable Markdown format.",
        max_new_tokens=2048
    )
    
    print("\n" + "="*80)
    print("EXTRACTED TEXT (Markdown):")
    print("="*80)
    print(extracted_text)
    
except Exception as e:
    print(f"Error loading sample image: {e}")

## 7. Advanced: Structured Data Extraction

Test extracting structured JSON data from documents.

In [ ]:
# Extract structured invoice data as JSON
if Path(invoice_path).exists():
    invoice_image = load_image(invoice_path)
    
    json_prompt = """
Extract all information from this invoice and return it as a JSON object with the following structure:
{
  "invoice_number": "",
  "invoice_date": "",
  "vendor": {
    "name": "",
    "address": ""
  },
  "customer": {
    "name": "",
    "address": ""
  },
  "line_items": [
    {
      "description": "",
      "quantity": 0,
      "unit_price": 0.0,
      "total": 0.0
    }
  ],
  "subtotal": 0.0,
  "tax": 0.0,
  "total": 0.0
}
"""
    
    print("Extracting structured JSON data...")
    json_result = extract_text_from_document(
        image=invoice_image,
        prompt=json_prompt,
        max_new_tokens=2048,
        temperature=0.1  # Lower temperature for more deterministic output
    )
    
    print("\n" + "="*80)
    print("EXTRACTED STRUCTURED DATA:")
    print("="*80)
    print(json_result)
    
    # Try to parse as JSON
    try:
        # Extract JSON from markdown code block if present
        if "```json" in json_result:
            json_str = json_result.split("```json")[1].split("```")[0].strip()
        elif "```" in json_result:
            json_str = json_result.split("```")[1].split("```")[0].strip()
        else:
            json_str = json_result
        
        parsed_json = json.loads(json_str)
        print("\n✓ Successfully parsed as JSON")
        print("\nParsed data:")
        print(json.dumps(parsed_json, indent=2))
    except json.JSONDecodeError as e:
        print(f"\n⚠ Could not parse as JSON: {e}")
        print("The model may need additional prompt engineering for strict JSON output.")
else:
    print(f"Invoice image not found: {invoice_path}")

## 8. Performance Benchmarking

In [ ]:
import time

# Benchmark extraction speed
if Path(invoice_path).exists():
    invoice_image = load_image(invoice_path)
    
    print("Running performance benchmark (3 iterations)...\n")
    
    times = []
    for i in range(3):
        start_time = time.time()
        
        result = extract_text_from_document(
            image=invoice_image,
            prompt="Convert this invoice to Markdown.",
            max_new_tokens=1024
        )
        
        elapsed = time.time() - start_time
        times.append(elapsed)
        
        print(f"Iteration {i+1}: {elapsed:.2f}s ({len(result)} chars)")
    
    print(f"\nAverage time: {sum(times)/len(times):.2f}s")
    print(f"Min time: {min(times):.2f}s")
    print(f"Max time: {max(times):.2f}s")
else:
    print(f"Invoice image not found: {invoice_path}")

## 9. Summary and Observations

### Model Capabilities:
- ✅ Document text extraction to Markdown
- ✅ Question answering about document content
- ✅ Structured data extraction (with prompt engineering)
- ✅ Support for various document types (invoices, statements, receipts)

### Performance Notes:
- Model size: 258M parameters (lightweight)
- Best on GPU with float16 precision
- Reasonable inference speed for a local model

### Use Cases for Integration:
1. **Document preprocessing** - Convert PDFs/images to Markdown before LLM processing
2. **Structured extraction** - Extract invoice/receipt data with schema guidance
3. **Document Q&A** - Interactive document understanding
4. **Batch processing** - Efficient processing of document archives

## 10. Cleanup

In [ ]:
# Free up GPU memory if needed
import gc

del model
del processor
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("✓ GPU memory cleared")
else:
    print("✓ Memory cleared")